In [63]:
import pickle
import torch
import torch.nn as nn
import pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.models import ERModel
from pykeen.nn import DistMultInteraction, TransEInteraction, RotatEInteraction
from pykeen.nn.representation import PartitionRepresentation, TransformedRepresentation, Embedding
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import MarginRankingLoss
from pykeen.evaluation import RankBasedEvaluator, SampledRankBasedEvaluator
from torch.optim import Adam, RMSprop, NAdam


In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
main_data = pd.read_csv('data/edges/IW_edges.csv')
main_data = main_data.astype(str)

triples = main_data[['id_entity_1', 'relation', 'id_entity_2']].values
triplet_data = TriplesFactory.from_labeled_triples(triples, create_inverse_triples=True)
training_set, testing_set, validation_set = triplet_data.split([0.8, 0.1, 0.1], random_state=17)

with open("data/dicts/bert_embeddings/protein_esm_35M.pkl", 'rb') as f:
    emb_prot = pickle.load(f)

with open("data/dicts/bert_embeddings/sm_chemberta_10M_MTR.pkl", 'rb') as f:
    emb_sm = pickle.load(f)

with open("data/dicts/bert_embeddings/rna_berta.pkl", 'rb') as f:
    emb_rna = pickle.load(f)

emb_dna = {}
files = ['data/dicts/bert_embeddings/dnabert_DNA.pkl', 'data/dicts/bert_embeddings/dnabert_NucleicAmbigous.pkl',
         'data/dicts/bert_embeddings/dnabert_NucleicMixed.pkl']
for file in files:
    with open(file, 'rb') as f:
        data = pickle.load(f)
        emb_dna.update(data)

emb_prot = {str(k): v for k, v in emb_prot.items()}
emb_sm   = {str(k): v for k, v in emb_sm.items()}
emb_rna  = {str(k): v for k, v in emb_rna.items()}
emb_dna  = {str(k): v for k, v in emb_dna.items()}

In [6]:
ent2id = triplet_data.entity_to_id   # dict[str,int]
num_entities = triplet_data.num_entities

assignment = torch.full((num_entities,), -1, dtype=torch.long, device=device)
base_ids   = torch.full((num_entities,), -1, dtype=torch.long, device=device)

def ids_in_tf(emb_dict):
    # возвращаем пары (entity_id, embedding_tensor) только если raw_id есть в TriplesFactory
    return [(ent2id[rid], v) for rid, v in emb_dict.items() if rid in ent2id]

In [67]:
N = 64

emb_by_type = [emb_prot, emb_sm, emb_dna, emb_rna]  # твои 4 dict'а
bases = []

for t, d in enumerate(emb_by_type):
    pairs = ids_in_tf(d)
    if len(pairs) == 0:
        raise ValueError(f"type {t}: no entities matched TriplesFactory labels")

    # важно: фиксируем порядок (по entity_id)
    pairs.sort(key=lambda x: x[0])
    ent_ids = torch.tensor([p[0] for p in pairs], dtype=torch.long, device=device)
    E = torch.stack([p[1] for p in pairs], dim=0).to(device)  # [num_t, dim_t]
    dim_t = E.shape[1]

    # база-таблица
    base = Embedding(max_id=E.shape[0], shape=(dim_t,)).to(device)
    with torch.no_grad():
        base._embeddings.weight.copy_(E)
    base._embeddings.weight.requires_grad_(False)

    # проекция в N
    rep_t = TransformedRepresentation(
        base=base,
        transformation=nn.Sequential(
            nn.Linear(dim_t, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, N),
        ).to(device),
    )
    bases.append(rep_t)

    # заполнение assignment/base_ids для сущностей этого типа
    assignment[ent_ids] = t
    base_ids[ent_ids] = torch.arange(E.shape[0], device=device)

In [68]:
assignment2 = torch.stack([assignment, base_ids], dim=1)  # [num_entities, 2]

ent_rep = PartitionRepresentation(
    assignment=assignment2,
    bases=bases,
    shape=(N,),
).to(device)

model = ERModel(
    triples_factory=triplet_data,
    interaction=DistMultInteraction,
    entity_representations=ent_rep,
    relation_representations_kwargs=dict(embedding_dim=N),
    random_seed=17
).to(device)

In [69]:
LR = 1e-3
MARGIN = 1.1
WEIGHT = 1e-3
EPOCHS = 5
BATCH_SIZE = 4096
NUM_NEGS_PER_POS = 15

loss_function = MarginRankingLoss(margin=MARGIN)

optimizer = Adam(params=model.parameters(), lr=LR)

training_loop = SLCWATrainingLoop(
    model=model,
    triples_factory=training_set,
    optimizer=optimizer,
    negative_sampler='pseudotyped',
    negative_sampler_kwargs=dict(
        num_negs_per_pos=NUM_NEGS_PER_POS
    )
)

training_loop.train(
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    triples_factory=training_set,
    use_tqdm_batch=False,
)

evaluator = RankBasedEvaluator()

model_results = evaluator.evaluate(
    model=model,
    mapped_triples=testing_set.mapped_triples[:1500].to(device),
    additional_filter_triples=[
        training_set.mapped_triples.to(device),
        validation_set.mapped_triples.to(device),
    ],
)

metrics = model_results.to_df()
metrics = metrics[(metrics['Side'] == 'both') & (metrics['Rank_type'] == 'realistic')]
metrics

Training epochs on cuda:0: 100%|██████████| 5/5 [01:35<00:00, 19.10s/epoch, loss=0.323, prev_loss=0.332]
Evaluating on cuda:0: 100%|██████████| 1.50k/1.50k [00:13<00:00, 112triple/s] 


,Side,Rank_type,Metric,Value
5,both,realistic,variance,7.770119e+09
14,both,realistic,adjusted_arithmetic_mean_rank,1.324030e-01
23,both,realistic,inverse_arithmetic_mean_rank,2.827759e-05
32,both,realistic,standard_deviation,8.814827e+04
41,both,realistic,z_arithmetic_mean_rank,8.230756e+01
50,both,realistic,inverse_median_rank,1.779993e-04
59,both,realistic,median_rank,5.618000e+03
68,both,realistic,z_geometric_mean_rank,5.337257e+01
77,both,realistic,arithmetic_mean_rank,3.536369e+04
86,both,realistic,inverse_geometric_mean_rank,1.965348e-04
